# 03 - Multi-Model Benchmarking & Clinical Evaluation
**Sepsis Clinical Decision Support System (CDSS)**  
*Script Source: `models_comparison/compare_models.py` & `optimal_15_features/compare_models_15.py`*

---

### Objectives:
1. **Model Architectures**: Compare three foundational paradigms:
   - **Logistic Regression**: Scaled linear baseline
   - **Single Decision Tree**: Interpretable white-box clinical decision tree
   - **Random Forest Classifier**: Non-linear ensemble of 100 decorrelated trees
2. **5-Fold Stratified Cross-Validation**: Benchmark out-of-fold AUROC stability.
3. **Comprehensive Performance Metrics**: Accuracy, Precision, Recall/Sensitivity, Specificity, F1-Score, ROC-AUC, PR-AUC, and Latency.
4. **Diagnostic Visualizations**:
   - Receiver Operating Characteristic (ROC) curves
   - Precision-Recall (PR) curves
   - Side-by-side Confusion Matrix heatmaps
5. **Clinical Trade-off Analysis**: Interpretability vs discrimination in clinical workflows.

In [ ]:
# Ensure working directory is the project root regardless of how the notebook is opened
import os
import sys

if os.path.basename(os.getcwd()) in ['jupyter_notebooks', 'notebooks']:
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Active Project Root: {os.getcwd()}")

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
import joblib

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries imported successfully!")

## Step 1: Dataset Ingestion & Feature Selection
We load the dataset and select the 15 optimal features.  
*(Note: If working on a lower-memory machine, you can optionally sample records; on a standard workstation, all rows are loaded)*.

In [ ]:
csv_paths = ['physionet_sepsis_processed.csv', 'physionet_sepsis_combined.csv']
dataset_path = next((p for p in csv_paths if os.path.exists(p)), None)

if not dataset_path:
    raise FileNotFoundError("Could not find sepsis dataset CSV file.")

print(f"Loading dataset from: {dataset_path}")
df = pd.read_csv(dataset_path)

OPTIMAL_FEATURES = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp",
    "WBC", "Glucose", "Lactate", "Creatinine", "Platelets",
    "Age", "Gender", "ICULOS"
]

X = df[OPTIMAL_FEATURES].copy()
y = df['SepsisLabel'].values

# Impute with column medians
medians = X.median().to_dict()
X_imputed = X.fillna(medians)

print(f"Dataset Shape: {X_imputed.shape[0]:,} rows x {X_imputed.shape[1]} features")
print(f"Sepsis Class Prevalence: {y.sum():,} positive records ({y.mean()*100:.2f}%)")

## Step 2: Stratified Partitioning
80% Training partition, 20% Testing partition stratified on `y`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.20, random_state=42, stratify=y
)

# Standardized features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Records: {len(X_train):,}")
print(f"Testing Records:  {len(X_test):,}")

## Step 3: Model Training & Latency Profiling
We train Logistic Regression, Decision Tree, and Random Forest models with balanced class weights.

In [ ]:
models = {}

# 1. Logistic Regression
print("1/3 Training Logistic Regression...")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
t0 = time.time()
lr.fit(X_train_scaled, y_train)
lr_time = time.time() - t0
models['Logistic Regression'] = {'model': lr, 'is_scaled': True, 'train_time': lr_time}

# 2. Decision Tree
print("2/3 Training Decision Tree Classifier...")
dt = DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42)
t0 = time.time()
dt.fit(X_train, y_train)
dt_time = time.time() - t0
models['Decision Tree'] = {'model': dt, 'is_scaled': False, 'train_time': dt_time}

# 3. Random Forest
print("3/3 Training Random Forest Classifier...")
rf = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1)
t0 = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - t0
models['Random Forest'] = {'model': rf, 'is_scaled': False, 'train_time': rf_time}

print("All models trained successfully!")

## Step 4: 5-Fold Stratified Cross-Validation
We evaluate out-of-fold ROC-AUC across 5 folds to verify that models do not suffer from split variance.

In [ ]:
print("Running 5-Fold Stratified Cross-Validation on training set (sampled 50k for speed)...")
sample_indices = np.random.RandomState(42).choice(len(X_train), size=min(50000, len(X_train)), replace=False)
X_cv = X_train.iloc[sample_indices]
y_cv = y_train[sample_indices]
X_cv_scaled = X_train_scaled[sample_indices]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, item in models.items():
    clf = item['model']
    X_target = X_cv_scaled if item['is_scaled'] else X_cv
    scores = cross_val_score(clf, X_target, y_cv, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = f"{scores.mean():.4f} (+/- {scores.std():.4f})"
    print(f" - {name:20s}: CV ROC-AUC = {cv_results[name]}")

## Step 5: Comprehensive Performance Comparison Table
Evaluating on the held-out test partition across Sensitivity/Recall, Specificity, F1-Score, ROC-AUC, and PR-AUC.

In [ ]:
results = []
predictions_dict = {}
probabilities_dict = {}

for name, item in models.items():
    clf = item['model']
    X_eval = X_test_scaled if item['is_scaled'] else X_test
    
    preds = clf.predict(X_eval)
    probs = clf.predict_proba(X_eval)[:, 1]
    
    predictions_dict[name] = preds
    probabilities_dict[name] = probs
    
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall / Sensitivity': recall_score(y_test, preds),
        'Specificity': tn / (tn + fp),
        'F1-Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probs),
        'PR-AUC': average_precision_score(y_test, probs),
        '5-Fold CV ROC-AUC': cv_results.get(name, 'N/A'),
        'Train Time (s)': round(item['train_time'], 2)
    })

comparison_df = pd.DataFrame(results).set_index('Model')
print("--- SUMMARY BENCHMARK TABLE ---")
comparison_df.round(4)

## Step 6: Diagnostic Visualizations
### 6.1 ROC and Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {'Logistic Regression': '#e67e22', 'Decision Tree': '#2ecc71', 'Random Forest': '#2980b9'}

# ROC Curves
for name, probs in probabilities_dict.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_val = roc_auc_score(y_test, probs)
    axes[0].plot(fpr, tpr, lw=2.2, color=colors[name], label=f"{name} (AUROC = {auc_val:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Baseline (AUC = 0.500)')
axes[0].set_title('Receiver Operating Characteristic (ROC) Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Sensitivity / Recall)')
axes[0].legend(loc='lower right', frameon=True)

# Precision-Recall Curves
baseline_prec = y_test.mean()
for name, probs in probabilities_dict.items():
    prec, rec, _ = precision_recall_curve(y_test, probs)
    pr_auc_val = average_precision_score(y_test, probs)
    axes[1].plot(rec, prec, lw=2.2, color=colors[name], label=f"{name} (PR-AUC = {pr_auc_val:.3f})")

axes[1].axhline(baseline_prec, color='black', linestyle='--', label=f'Baseline ({baseline_prec:.3f})')
axes[1].set_title('Precision-Recall (PR) Curve Comparison', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision (Positive Predictive Value)')
axes[1].legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

### 6.2 Confusion Matrix Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, preds) in zip(axes, predictions_dict.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Pred Non-Sepsis', 'Pred Sepsis'],
                yticklabels=['Actual Non-Sepsis', 'Actual Sepsis'])
    ax.set_title(f'{name} Confusion Matrix', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## Step 7: Clinical Insights & Comparison Analysis
1. **Variance Reduction**: Single Decision Trees overfit local noise in vital sign spikes, leading to lower Specificity and high False Alarm rates. Random Forest's bagging ensemble across 100 decorrelated trees reduces variance substantially.
2. **Handling Extreme Class Imbalance**: Random Forest achieves the highest PR-AUC (~0.33+ vs ~0.11 for single tree), minimizing clinician alert fatigue.
3. **Selected CDSS Architecture**: Random Forest delivers optimal sensitivity and discrimination for the CDSS inference engine.